# label_scored_trec21 — the ordering-signal experiment (dev = TREC21)

Design §8d: the gate proved a whole-doc labeler beats clf, but the **3-way grade saturates** — clf's
tiebreak orders the top-10 and wastes ~0.29 NDCG@10 (oracle-within-pred2 0.79 vs actual 0.505). Fix:
the labeler emits a **continuous 0-100 relevance score**; rank by it.

**This runs on TREC21, not TREC22** (§4 test hygiene — TREC22 is spent once, at the end). no-CoT base
(cheaper/faster, and ≥ CoT on NDCG per §8c). `PROBE_N=10` first (~$8), then lift to all 75.

## Setup (Colab — CPU fine; clf cached, labeler is API)

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q anthropic nest_asyncio pandas tqdm datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, time, re, asyncio
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval, full_blob, ndcg_at_k
cfg = ExperimentConfig(data_root=DATA_ROOT)
SPLIT = 'trec21'               # DEV — never TREC22 during development
CLAUDE_MODEL = 'claude-opus-4-8'
PRICE_IN, PRICE_OUT = 5.0, 25.0
LABEL_W = 100
PROBE_N = 10                   # None = all TREC21 topics
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    assert os.environ.get('ANTHROPIC_API_KEY'), 'set ANTHROPIC_API_KEY (Colab secret or os.environ)'
print('key set:', bool(os.environ.get('ANTHROPIC_API_KEY')))

## Load corpus, TREC21 qrels, clf top-100

In [ ]:
corpus_ids, corpus_fields = load_corpus(cfg)
id2fields = dict(zip(corpus_ids, corpus_fields))
ev = load_eval(cfg, [SPLIT])[SPLIT]
rel, topic2text = ev['rel_dict'], ev['topic2text']
topics = [t for t in rel if t in topic2text]
if PROBE_N:
    topics = topics[:PROBE_N]; print(f'PROBE: {len(topics)} topics (set PROBE_N=None to finish; resumes)')
pool = json.load(open(cfg.path('data/pool_R.json')))[SPLIT]
ce = np.load(cfg.ce_cache_path('clf'), allow_pickle=True)['d'].item()
clf_scores = {t: {d: ce[(SPLIT, t, d)][0] for d in pool[t] if (SPLIT, t, d) in ce} for t in topics}
clf_order  = {t: sorted(pool[t], key=lambda d: clf_scores[t].get(d, -1.0), reverse=True) for t in topics}
print(f'{len(topics)} topics | {len(topics)*LABEL_W} calls | '
      f'clf-top100 oracle (ceiling) = '
      f'{np.mean([ndcg_at_k(sorted(clf_order[t][:LABEL_W], key=lambda d: rel[t].get(d,0), reverse=True)+clf_order[t][LABEL_W:], rel[t]) for t in topics]):.3f}')

## Scored labeler (no-CoT) — 0-100 relevance, ranked directly

A continuous score gives intra-bucket ordering the 3-way grade can't. Excluded-but-on-topic sits
mid-scale so it still outranks off-topic trials (matches qrel gains 2>1>0).

In [ ]:
SYSTEM = ('You are a clinical trial relevance assessor. Score how well a clinical trial matches a '
          'patient, for ranking trials by relevance to that patient.')
def make_prompt(topic_text, doc_text):
    return (f'Patient case:\n{topic_text}\n\nClinical trial:\n{doc_text}\n\n'
            'Give a single relevance score from 0 to 100:\n'
            '90-100 = clearly eligible: trial targets the patient\'s condition AND the patient meets its criteria.\n'
            '30-60  = on-topic but the patient appears excluded by the criteria.\n'
            '0-15   = not relevant: trial does not target the patient\'s condition.\n'
            'Use the full range to reflect confidence. Respond with only: SCORE: N')
def parse_score(text):
    m = re.search(r'SCORE:\s*(\d{1,3})', text)
    if m: return max(0, min(100, int(m.group(1))))
    m = re.findall(r'\d{1,3}', text)
    return max(0, min(100, int(m[-1]))) if m else None

In [ ]:
import anthropic, nest_asyncio
nest_asyncio.apply()
aclient = anthropic.AsyncAnthropic(); MAX_CONCURRENT = 15
async def score_one(topic_text, doc_text, sem):
    kw = dict(model=CLAUDE_MODEL, system=SYSTEM, thinking={'type': 'disabled'}, max_tokens=64,
              messages=[{'role': 'user', 'content': make_prompt(topic_text, doc_text)}])
    async with sem:
        for attempt in range(4):
            try:
                msg = await aclient.messages.create(**kw)
                txt = ''.join(b.text for b in msg.content if b.type == 'text')
                s = parse_score(txt)
                return (s if s is not None else 0, msg.usage.input_tokens, msg.usage.output_tokens, s is not None)
            except Exception as e:
                if attempt == 3:
                    print('give up:', repr(e)[:120]); return 0, 0, 0, False
                await asyncio.sleep(2 ** attempt)
async def score_topic(t, docs):
    sem = asyncio.Semaphore(MAX_CONCURRENT)
    tasks = [score_one(topic2text[t], full_blob(id2fields[d], cfg), sem) for d in docs]
    return list(zip(docs, await asyncio.gather(*tasks)))

In [ ]:
_t = topics[0]
for _d in clf_order[_t][:3]:
    (s, i, o, ok), = [r for _, r in asyncio.run(score_topic(_t, [_d]))]
    print(f'score={s:3d} ok={ok} in={i} out={o}  gold={rel[_t].get(_d,0)}')

## Cost estimate (count_tokens) before spending

In [ ]:
import random, anthropic as _a
_sync = _a.Anthropic()
samp = [(t, d) for t in topics for d in clf_order[t][:LABEL_W]]
random.Random(0).shuffle(samp)
mean_in = np.mean([_sync.messages.count_tokens(model=CLAUDE_MODEL, system=SYSTEM,
        messages=[{'role': 'user', 'content': make_prompt(topic2text[t], full_blob(id2fields[d], cfg))}]).input_tokens
    for t, d in samp[:20]])
n = len(topics) * LABEL_W
est = n * (mean_in * PRICE_IN + 6 * PRICE_OUT) / 1e6
print(f'mean in-tok {mean_in:.0f} | {n} calls | EST ~${est:.2f} for {len(topics)} topics')

## Run (resumable) → score every clf top-100 doc

In [ ]:
spath = cfg.path(f'data/scored_{SPLIT}.jsonl')
scores, usage, fails, lat, done = {}, [0, 0], 0, {}, set()
if os.path.exists(spath):
    for l in open(spath):
        r = json.loads(l); scores[(r['topic_id'], r['doc_id'])] = r['score']
        usage[0] += r['in_tok']; usage[1] += r['out_tok']; fails += 0 if r.get('ok', True) else 1
        done.add(r['topic_id'])
    print('resumed topics:', len(done))
with open(spath, 'a') as f:
    for t in tqdm(topics, desc=f'score {SPLIT}'):
        if t in done: continue
        t0 = time.time(); res = asyncio.run(score_topic(t, clf_order[t][:LABEL_W])); lat[t] = time.time()-t0
        for d, (s, i, o, ok) in res:
            scores[(t, d)] = s; usage[0]+=i; usage[1]+=o; fails += 0 if ok else 1
            f.write(json.dumps({'topic_id': t, 'doc_id': d, 'score': s, 'in_tok': i, 'out_tok': o, 'ok': ok})+'\n')
        f.flush()
print('done. tokens:', usage, '| parse-fails:', fails)

## Result — does score-ordering capture the gap?

In [ ]:
def rank_scored(t, W=LABEL_W):
    top = clf_order[t][:W]
    top_sorted = sorted(top, key=lambda d: (scores.get((t, d), 0), clf_scores[t].get(d, 0)), reverse=True)
    return top_sorted + clf_order[t][W:]
def mean_ndcg(rank_fn): return float(np.mean([ndcg_at_k(rank_fn(t), rel[t]) for t in topics]))
clf_ndcg    = mean_ndcg(lambda t: clf_order[t])
scored_ndcg = mean_ndcg(rank_scored)
oracle_ndcg = mean_ndcg(lambda t: sorted(clf_order[t][:LABEL_W], key=lambda d: rel[t].get(d,0), reverse=True)+clf_order[t][LABEL_W:])
cost = (usage[0]*PRICE_IN + usage[1]*PRICE_OUT)/1e6/len(topics)
captured = (scored_ndcg-clf_ndcg)/(oracle_ndcg-clf_ndcg) if oracle_ndcg>clf_ndcg else float('nan')
print(f'clf-alone      NDCG@10 = {clf_ndcg:.4f}')
print(f'scored labeler NDCG@10 = {scored_ndcg:.4f}   (${cost:.3f}/topic, fail {fails})')
print(f'oracle (top100 order)  = {oracle_ndcg:.4f}')
print(f'-> captured {captured:.0%} of the clf->oracle ordering gap')
pd.DataFrame([{'clf': round(clf_ndcg,4), 'scored': round(scored_ndcg,4), 'oracle': round(oracle_ndcg,4),
               'captured_frac': round(captured,3), '$/topic': round(cost,4)}]).to_csv(cfg.path(f'data/scored_result_{SPLIT}.csv'), index=False)

## Read
- **scored ≫ clf and near oracle** → the ordering fix is the win; this becomes the base labeler.
  Beating SOTA (0.6125) here on TREC21 dev is the green light to freeze and spend once on TREC22.
- **scored barely beats clf** → the labeler can't order finely; fall back to reflection on recall (§8d
  lever a) or CoT for higher bucket recall.
- Either way this is **dev (TREC21)** — tune here, freeze, then one TREC22 run for the headline.